In [2]:
import geopandas as gpd 
import pandas as pd
import matplotlib.pyplot as plt
import folium
from folium import plugins

#add the fire data from 2023
fire_2023 = pd.read_csv("../data/processed/fire_2023_clean.csv")

#Now we create point geometries
fire_2023_gdf = gpd.GeoDataFrame(
    fire_2023, geometry=gpd.points_from_xy(
        fire_2023.longitude, fire_2023.latitude),
    crs="EPSG:4326")

fire_2023_gdf.to_file("../data/processed/fire_2023_points.geojson", driver="GeoJSON")


#First we create a new Folium map, which is centered to Canada. 
canada_map = folium.Map(
    location = [60.03,-98.82],
    zoom_start =4,
    tiles = "CartoDB.DarkMatterNoLabels", 
)

#Heatmap for weighted fires

heat_data = [[point.xy[1][0], point.xy[0][0], weight] 
             for point, weight in zip (
                 fire_2023_gdf.geometry,
                 fire_2023_gdf["class_weight"])
                 ]

plugins.HeatMap(
    heat_data,
    radius=13,
    blur=5,
    min_opacity=0.25,
    name = "Wildfires in 2023",
    gradient={
        0.0: "#FFFACD",    
        0.25: "#FFFF00",  
        0.50: "#FFA500",
        0.75: "#FFA500", 
        1.0: "#8B0000"  }
).add_to(canada_map)

#Then we display the cities on to canada
#add the city data

selected_cities = pd.read_csv("../data/processed/selected_cities_canada.csv")

#Now we create point geometries
selected_cities_gdf = gpd.GeoDataFrame(
    selected_cities, geometry=gpd.points_from_xy(
        selected_cities.lng, selected_cities.lat),
    crs="EPSG:4326")

selected_cities_gdf.to_file("../data/processed/selected_cities_canada.geojson", driver="GeoJSON")

#make a buffer around the cities

#to make buffers we need the point projections in metric 
selected_cities_gdf = selected_cities_gdf.to_crs("EPSG:3347")

buffer_km = 200
cities_buffered = selected_cities_gdf.copy()

cities_buffered["geometry"] = cities_buffered.geometry.buffer(buffer_km * 1000)

#color pins > both in the same crs is necessary 
fire_2023_gdf = fire_2023_gdf.to_crs(cities_buffered.crs)

#spatial join 
joined_data = gpd.sjoin(
    cities_buffered, 
    fire_2023_gdf,
    how="left", #cities without fires should be visible
    predicate = "contains")

fire_per_city = (
    joined_data
        .groupby("city")["class_weight"]
        .sum()
        .reset_index()
        .rename(columns={"class_weight": "fire_score"}))

selected_cities_gdf = selected_cities_gdf.merge(fire_per_city, on="city", how="left").to_crs("EPSG:4326")
selected_cities_gdf["fire_score"] = selected_cities_gdf["fire_score"].fillna(0)

def impact(fire_score):
    if fire_score == 0:
        return "No impact"
    elif fire_score > 0 and fire_score < 20:
        return "Low impact"
    elif fire_score >= 20 and fire_score < 50:
        return "Moderate impact"
    elif fire_score >= 50 and fire_score < 256:
        return "High impact"
    else:
        return "Very high impact"

selected_cities_gdf["impact"] = selected_cities_gdf["fire_score"].apply(impact)

#color the pins 
impacted_cities_layer = folium.FeatureGroup(name = "Impacted Cities in 2023")
for city,row in selected_cities_gdf.iterrows(): 
    fire_score = row["fire_score"]
    
    if fire_score == 0:
        city_color = "#E0FFFF"
    elif fire_score > 0 and fire_score < 20:
        city_color = "#00CED1"
    elif fire_score >= 20 and fire_score < 50:
        city_color = "#4169E1"
    elif fire_score >= 50 and fire_score < 256:
        city_color = "#0000FF"
    else:
        city_color = "#191970"

    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius = 7,
        color = "black",
        weight = 1,
        fill = True,
        fill_color=city_color,
        fill_opacity = 1,
        tooltip= (f"Name: {row["city"]} <br>"
                  f"Country: {row["country"]}<br>"
                  f"Population: {row["population"]} <br>"
                  f"Impact: {row["impact"]}"),
    ).add_to(impacted_cities_layer),
    
impacted_cities_layer.add_to(canada_map)

In [3]:
import requests

#Now we proceed with coding the fire map with FIRMS Nasa data

map_key = "b73850f35cfea7f219a3969b6df1d9a6"
api_url = f"https://firms.modaps.eosdis.nasa.gov/usfs/api/area/csv/{map_key}/VIIRS_NOAA20_NRT/world/5/2026-05-12"

response = requests.get(api_url)
print(response.status_code)

if response.status_code == 200:
    fire_live_df = pd.read_csv(api_url)
    fire_live_df.to_csv("../data/processed/fire_live.csv", index=False)
    #to not load the whole world, we define a bounding box
    north = 73.226700
    south = 41.640078
    west  = -140.625000
    east  = -49.218750
    fire_live_df = fire_live_df[
        (fire_live_df["latitude"] >= south) &
        (fire_live_df["latitude"] <= north) &
        (fire_live_df["longitude"] >= west) &
        (fire_live_df["longitude"] <= east)
        ]
    print("success")
else:
    print("Error")

live_heatmap = [
    [row["latitude"], row["longitude"]]
    for _, row in fire_live_df.iterrows()
] 
plugins.HeatMap(
    live_heatmap,
    radius=13,
    blur=5,
    min_opacity=0.25,
    name = "Live wildfire",
    show = False,
    gradient={
        0.0: "#FFFACD",    
        0.25: "#FFFF00",  
        0.50: "#FFA500",
        0.75: "#FFA500", 
        1.0: "#8B0000"  }
).add_to(canada_map)

folium.LayerControl().add_to(canada_map)

200
success


In [4]:
canada_map